In [79]:
import pandas as pd
import scipy as sp
import numpy as np
from matplotlib import pyplot

Notes:
V242067: Post, who did the respondent vote for President?
V240002c: Pre-/Post-election interview completion status -- this is well good to use!
V241458x: Pre, respondent age on election day

What I care about:
V241501x: Pre, summary, respondent's self-identified race/ethnicity
V241567x: Pre, summary, total household income (6-category)
V241566x: Pre, summary, total household income (more categories)
V241039: Pre, for whom did R vote for President
V240108a: Full pre-election sample weights

In [80]:
# Loading in the dataset

anes2024_csv = "../../../dataset/data_2024/anes_timeseries_2024_csv_20250808.csv"
df2024 = pd.read_csv(anes2024_csv)

df2024_select = df2024[["V241501x", "V241567x", "V241566x", "V241039", "V240108a",
                        "V240002c", "V242067"]]
df2024_select

C:\Users\james\AppData\Local\Temp\ipykernel_5456\1533670907.py:4: DtypeWarning: Columns (0: V240101a, 1: V240101c, 2: V240101d, 3: V240103a, 4: V240103c, 5: V240103d, 6: V240104a, 7: V240104c, 8: V240104d, 9: V240105a, 10: V240105c, 11: V240105d, 12: V240106a, 13: V240106c, 14: V240106d, 15: V240108a, 16: V240108c, 17: V240108d, 18: V243002, 19: V243003, 20: V243004, 21: V243005, 22: V243006, 23: V243009, 24: V243050, 25: V243051, 26: V243052, 27: V243053, 28: V243056, 29: V243057, 30: V243058) have mixed types. Specify dtype option on import or set low_memory=False.
  df2024 = pd.read_csv(anes2024_csv)


,V241501x,V241567x,V241566x,V241039,V240108a,V240002c,V242067
0,3,5,27,-1,0.809064,2,2
1,4,5,26,-1,2.575472,2,-1
2,1,5,24,-1,0.798864,2,1
3,4,3,12,-1,0.244757,2,-1
4,1,3,15,-1,0.239209,2,2
...,...,...,...,...,...,...,...
5516,-8,2,5,-1,0.19183,1,-6
5517,1,3,16,-1,2.869063,2,1
5518,2,3,12,-9,2.092296,1,-6
5519,2,4,19,-1,2.826738,2,1


In [82]:
# Dictionaries for conversion

race_dict = { # V241501x
    -9: "Refused",
    -8: "Don’t Know",
    -4: "Error",
    1: "White", #"White, non-Hispanic",
    2: "Black", #"Black, non-Hispanic",
    3: "Hispanic",
    4: "Asian/Pacific-Islander", #"Asian or Native Hawaiian/other Pacific Islander, non-Hispanic",
    5: "Native American/Other", #"Native American/Alaska Native or other race, non-Hispanic",
    6: "Multiple Races", #"Multiple races, non-Hispanic",
}

household_income_6_cat_dict = { # V241567x
    -9: "Refused",
    -5: "Break off, sufficient partial",
    -4: "Error",
    1: "<$10k", #"Under $9,999",
    2: "$10k-30k", #"$10,000 to $29,999",
    3: "$30k-60k", #"$30,000 to $59,999",
    4: "$60k-100k", #"$60,000 to $99,999",
    5: "$100k-250k", #"$100,000 to $249,999",
    6: "$250k+", #"$250,000 or more",
}

# Okay, forget the 28-category one.

president_dict = { # V242067
    -9: "Refused",
    -1: "Inapplicable",
    1: "Harris", #"Kamala Harris",
    2: "Trump", #"Donald Trump",
    3: "RFK Jr.", #"Robert F. Kennedy, Jr.",
    4: "West", #"Cornel West",
    5: "Stein", #"Jill Stein",
    6: "Other", #"Another candidate {SPECIFY}"
}

complete_survey_dict = {
    1: 0,
    2: 1
}

In [83]:
# Making the dataset more readable
df2024_select["Race"] = df2024_select["V241501x"].map(race_dict)
df2024_select["Household Income"] = df2024_select["V241567x"].map(household_income_6_cat_dict)
df2024_select["President Vote"] = df2024_select["V242067"].map(president_dict)
df2024_select["Completed Survey"] = df2024_select["V240002c"].map(complete_survey_dict)

# Clean up 'Weights': replace empty strings with 0.
df2024_select["Weights"] = df2024_select["V240108a"]
df2024_select["Weights"] = df2024_select["Weights"].apply(lambda x: 0 if x == ' ' or x == '' else x)
df2024_select["Weights"] = df2024_select["Weights"].astype(float)

df2024_clean = df2024_select[["Race", "Household Income", "President Vote",
                              "Completed Survey", "Weights"]]

df2024_clean

#df2024_clean[df2024_clean["Weights"] == ' ']["Weights"]
#df2024_clean["President Vote"].value_counts()
#df2024_clean[df2024_clean["Weights"].isna()]

,Race,Household Income,President Vote,Completed Survey,Weights
0,Hispanic,$100k-250k,Trump,1,0.809064
1,Asian/Pacific-Islander,$100k-250k,Inapplicable,1,2.575472
2,White,$100k-250k,Harris,1,0.798864
3,Asian/Pacific-Islander,$30k-60k,Inapplicable,1,0.244757
4,White,$30k-60k,Trump,1,0.239209
...,...,...,...,...,...
5516,Don’t Know,$10k-30k,NaN,0,0.191830
5517,White,$30k-60k,Harris,1,2.869063
5518,Black,$30k-60k,NaN,0,2.092296
5519,Black,$60k-100k,Harris,1,2.826738


In [84]:
# Dealing with missing values
df2024_clean.dropna()
df2024_clean


# Order variables
incomes = ["<$10k", "$10k-30k", "$30k-60k", "$60k-100k", "$100k-250k", "$250k+",
           "Refused", "Break off, sufficient partial", "Error"]
df2024_clean["Household Income"] = pd.Categorical(values = df2024_clean["Household Income"],
                                                  categories = incomes, ordered = True)

df2024_relevant = df2024_clean[df2024_clean["Completed Survey"] == 1]

df2024_relevant

,Race,Household Income,President Vote,Completed Survey,Weights
0,Hispanic,$100k-250k,Trump,1,0.809064
1,Asian/Pacific-Islander,$100k-250k,Inapplicable,1,2.575472
2,White,$100k-250k,Harris,1,0.798864
3,Asian/Pacific-Islander,$30k-60k,Inapplicable,1,0.244757
4,White,$30k-60k,Trump,1,0.239209
...,...,...,...,...,...
5514,White,$100k-250k,Inapplicable,1,0.797417
5515,Hispanic,<$10k,Harris,1,0.129755
5517,White,$30k-60k,Harris,1,2.869063
5519,Black,$60k-100k,Harris,1,2.826738


In [85]:
# EDA
# Cross-Tabs of Race, Income, and President Choice
president_race_tab = pd.crosstab(index = df2024_relevant["Race"],
                                 columns = df2024_relevant["President Vote"],
                                 values = df2024_relevant["Weights"],
                                 aggfunc=sum, normalize="index").fillna(0)

# FYI, NaN values come from the aggregate function.
race_income_tab = pd.crosstab(index = df2024_relevant["Race"],
                              columns = df2024_relevant["Household Income"],
                              values = df2024_relevant["Weights"], 
                              aggfunc=sum, normalize="index").fillna(0)

race_income_tab_percents = race_income_tab.round(4) * 100
race_income_tab_percents

president_race_tab_percents = president_race_tab.round(4) * 100
president_race_tab_percents

President Vote,Harris,Inapplicable,Other,Refused,Stein,Trump,West
Race,,,,,,,
Asian/Pacific-Islander,47.93,22.88,1.34,0.00,0.22,27.35,0.27
Black,62.35,29.17,1.13,0.66,0.08,6.18,0.42
Hispanic,39.23,39.40,0.66,0.45,1.55,18.52,0.19
Multiple Races,34.58,32.40,2.78,0.29,0.62,28.70,0.63
Native American/Other,23.54,44.32,4.19,0.00,0.00,27.94,0.00
Refused,20.93,25.04,0.00,9.08,0.00,44.24,0.71
White,35.47,22.56,1.45,1.15,0.49,38.78,0.09


Notes:
"Inapplicable" is vague. Would need someone to filter to the rows that positively answered whether or not they voted, and classify based on that. That's this variable (Post: did R vote in the 2024 election): V242065
Then, we can have "didn't vote" as a category. I think it'd be useful, because low turnout was a feature of the 2024 Presidential Election.

In [ ]:
# Analyze relationship between race and Presidential vote choice.
# Simplifying Data to only people that answered with a candidate.
df2024_simplified = df2024_relevant[df2024_relevant["Race"] != "Refused"]

def collapse_third_party(vote):
    if vote in ["West", "Stein", "Other"]:
        return "Other"
    else:
        return vote

df2024_simplified["President Vote"] = df2024_simplified["President Vote"].apply(collapse_third_party)

df2024_simplified = df2024_simplified[df2024_simplified["President Vote"] != "Refused"]
df2024_simplified = df2024_simplified[df2024_simplified["President Vote"] != "Inapplicable"]

df2024_simplified
#df2024_simplified["President Vote"].value_counts()

,Race,Household Income,President Vote,Completed Survey,Weights
0,Hispanic,$100k-250k,Trump,1,0.809064
2,White,$100k-250k,Harris,1,0.798864
4,White,$30k-60k,Trump,1,0.239209
6,White,$30k-60k,Trump,1,0.290135
10,White,$100k-250k,Harris,1,0.940042
...,...,...,...,...,...
5507,White,$60k-100k,Other,1,3.652063
5515,Hispanic,<$10k,Harris,1,0.129755
5517,White,$30k-60k,Harris,1,2.869063
5519,Black,$60k-100k,Harris,1,2.826738


In [102]:
# Cross-Tab with simplified, positive voter dataset
race_president_cross_tab = pd.crosstab(index = df2024_simplified["Race"],
                                 columns = df2024_simplified["President Vote"],
                                 values = df2024_simplified["Weights"],
                                 aggfunc=sum, normalize="index").fillna(0)

race_president_cross_tab_percents = race_president_cross_tab.round(4) * 100
race_president_cross_tab_percents

President Vote,Harris,Other,Trump
Race,,,
Asian/Pacific-Islander,62.15,2.38,35.47
Black,88.87,2.33,8.80
Hispanic,65.23,3.99,30.79
Multiple Races,51.37,5.99,42.64
Native American/Other,42.28,7.53,50.19
White,46.50,2.67,50.83


In [ ]:
# Chi^2 Test on this Cross-Tabulation

# We need to actually remake the cross-tab, because Chi^2 cares about counts.
race_president_cross_tab_count = pd.crosstab(index = df2024_simplified["Race"],
                                 columns = df2024_simplified["President Vote"],
                                 values = df2024_simplified["Weights"],
                                 aggfunc=sum).fillna(0)


race_president_test = sp.stats.chi2_contingency(race_president_cross_tab_count)
p_value = race_president_test.pvalue
print(f"P-Value: {float(p_value)}")

race_president_cross_tab_count
# The values in this cross-tab are floats due to weighting, but they still function as
# cross-category frequencies.

P-Value: 5.313565461524059e-54


President Vote,Harris,Other,Trump
Race,,,
Asian/Pacific-Islander,105.033297,4.014358,59.938848
Black,332.888678,8.738046,32.972940
Hispanic,256.988709,15.701025,121.302511
Multiple Races,74.991483,8.740201,62.245260
Native American/Other,4.740423,0.844645,5.627008
White,1060.584502,60.941264,1159.539760


Well, based on this Chi^2 test, we can conclude that there is a statistically significant difference between the frequences of votes for Harris, Trump, and other candidates based on racial group of the voter. The p-value is well less than the statistical significance threshold of alpha = 0.05

It should be noted that there are possible confounders to this relationship, namely income group and age.

Outliers here would be votes for other candidates besides Harris and Trump, which make up a substantial minority of voters.
We filtered out non-voters and people who refused to or otherwise did not answer the questions about race and who they voted for in the Presidential Election. We also (necessarily) only considered survey entries that completed the post-election survey (to indicate who they voted for).
Imbalance of demographic groups should be mitigated for by the row weights, and the ANES has assured that its samples are representative (I think).

Based on the cross-tab, it would appear that Asians, Hispanics, and especially African Americans were more likely to vote for Harris than Trump. People of multiple races were more likely to vote for Harris than Trump, but by less of a significant margin. Whites and Native Americans were more likely to vote for Trump, but also by less of a significant margin. It should be noted that the Native American sample is small, and so perhaps biased. Whites are well overrepresented in this dataset, with Blacks, Hispanics, and Asians being the next highest groups.

Notes for peers:
- If you're using post-election variables, make sure to filter the dataset to only rows where people completed the post-election survey.
- Make sure to use the weights for the survey rows (you can look up how to use them with Pandas's operations, or look at my examples). The basic intuition is that each weight represents the number of people that the row's response should account for. It has no relation to column values.
- I did a cross-tabulation and a Chi^2 Test with two categorical variables (race and Presidential choice).
- We should perhaps do a continuous versus continuous and categorical versus continuous relationship. If you can, try and focus it on the vote choice or general preference for President in the 2024 election. The dataset is large. There is a lot to analyze, but we'll need to focus on some particular result.
- Age and the "Feeling Thermometers" are some continuous variables we can use. Income group and political party are good categorical variables to look at.
- Make sure to look at the codebook to find the variable names and value encodings. In my .ipynb file, I have some of these in there already, if you're interested in them. Make sure to recode the encoded values to something meaningful, if necessary.